# JSEP — Vehicle Detector Training
> **Jakarta Smart Enforcement Platform** | AI Open Innovation Challenge 2026  
> PRD Section 11.4 — YOLO fine-tuning on 8 Indonesian traffic classes
>
> **Classes**: car, motorcycle, truck, bus, angkot, bajaj, bicycle, pedestrian  
> **Target KPI**: mAP50 > 0.90 (KPI-01), Precision > 0.85 (KPI-04)


## 0 — Environment Setup


In [ ]:
import subprocess, os

# Check GPU
!nvidia-smi

# Install dependencies
!pip install -q ultralytics>=8.3 roboflow pyyaml

print('Setup complete.')

## 1 — Configuration


In [ ]:
from pathlib import Path
import yaml, os

# ── Paths ──────────────────────────────────────────────────────────────
KAGGLE_WORKING = Path('/kaggle/working')
DATASETS_DIR   = KAGGLE_WORKING / 'datasets'
RUNS_DIR       = KAGGLE_WORKING / 'runs'
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

# ── Model base (RULE-07: YOLO26 primary, YOLOv11 fallback) ─────────────
# Kaggle does not have yolo26 weights by default — use yolo26n.pt as base
# and rename convention; swap to yolo26 when weights are available
MODEL_BASE = 'yolo26n.pt'   # swap to 'yolo26n.pt' when available

# ── 8 JSEP Classes (PRD Section 11.2) ──────────────────────────────────
VEHICLE_CLASSES = [
    'car', 'motorcycle', 'truck', 'bus',
    'angkot', 'bajaj', 'bicycle', 'pedestrian'
]

# ── Training Hyperparameters (PRD Section 11.4) ─────────────────────────
TRAIN_CFG = dict(
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,          # P100/T4: 16; A100: 32
    patience  = 25,
    device    = 0,
    project   = str(RUNS_DIR / 'jsep'),
    name      = 'vehicle_detector_v1',
    exist_ok  = True,
    val       = True,
    # PRD Section 11.3 augmentation
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    degrees   = 5.0,
    translate = 0.1,
    scale     = 0.5,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.1,
    copy_paste= 0.1,
    erasing   = 0.4,         # rain/night simulation (PRD FR-DET-08)
)

print('Config ready.')
print(f'Classes ({len(VEHICLE_CLASSES)}): {VEHICLE_CLASSES}')

## 2 — Dataset Download (Roboflow)


In [ ]:
# ── Roboflow API key from Kaggle Secret: ROBOFLOW_API_KEY ──────────────
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
rf_api_key = secrets.get_secret('ROBOFLOW_API_KEY')

from roboflow import Roboflow
rf = Roboflow(api_key=rf_api_key)

# PRD Section 11.1 — Vehicle detection datasets (Indonesian traffic)
DATASETS = [
    # (workspace,          project,                          version, description)
    ('roboflow-100',       'vehicles-q0x2v',                 2,       'General vehicles ~3k'),
    ('roboflow-100',       'motorcycle-vs-car-gmkqm',        1,       'Moto/car discrimination'),
    ('vehicle-detection-auvo9', 'vehicle-detection-aknnz',   2,       'Urban traffic mix'),
]

downloaded = []
for workspace, project_name, version, desc in DATASETS:
    dest = DATASETS_DIR / project_name
    if dest.exists():
        print(f'  Already exists: {project_name}')
        downloaded.append(dest)
        continue
    try:
        proj = rf.workspace(workspace).project(project_name)
        proj.version(version).download('yolov8', location=str(dest))
        print(f'  ✓ Downloaded: {project_name} ({desc})')
        downloaded.append(dest)
    except Exception as e:
        print(f'  ✗ Failed {project_name}: {e}')

print(f'\nTotal datasets downloaded: {len(downloaded)}')

In [ ]:
# ── Fallback: Use COCO subset if no Roboflow datasets downloaded ────────
if not downloaded:
    print('No Roboflow datasets. Downloading COCO8 as smoke-test fallback...')
    !yolo data download dataset=coco8
    # After COCO8 download, point to its yaml (8-class remap happens below)
    fallback_yaml = Path('/kaggle/working/datasets/coco8/coco8.yaml')
    FALLBACK_MODE = True
else:
    FALLBACK_MODE = False

print(f'Fallback mode: {FALLBACK_MODE}')

## 3 — Merge Datasets & Build YAML


In [ ]:
import shutil
from pathlib import Path

MERGED_DIR = DATASETS_DIR / 'merged'
(MERGED_DIR / 'train' / 'images').mkdir(parents=True, exist_ok=True)
(MERGED_DIR / 'train' / 'labels').mkdir(parents=True, exist_ok=True)
(MERGED_DIR / 'valid' / 'images').mkdir(parents=True, exist_ok=True)
(MERGED_DIR / 'valid' / 'labels').mkdir(parents=True, exist_ok=True)

train_imgs, val_imgs = [], []

for ds in downloaded:
    # Roboflow YOLOv8 format: train/images, valid/images
    for split in ['train', 'valid']:
        img_dir = ds / split / 'images'
        lbl_dir = ds / split / 'labels'
        if img_dir.exists():
            target_list = train_imgs if split == 'train' else val_imgs
            for img in img_dir.glob('*.*'):
                target_list.append(img)

# Write merged dataset.yaml
train_dirs = list(set(str(p.parent) for p in train_imgs))
val_dirs   = list(set(str(p.parent) for p in val_imgs))

merged_yaml_path = MERGED_DIR / 'dataset.yaml'
merged_yaml = {
    'path' : str(MERGED_DIR),
    'train': train_dirs if train_dirs else '../train/images',
    'val'  : val_dirs   if val_dirs   else '../valid/images',
    'nc'   : len(VEHICLE_CLASSES),
    'names': VEHICLE_CLASSES,
}

with open(merged_yaml_path, 'w') as f:
    yaml.dump(merged_yaml, f, sort_keys=False)

print(f'Train images: {len(train_imgs)}')
print(f'Val   images: {len(val_imgs)}')
print(f'Merged YAML : {merged_yaml_path}')
print(yaml.dump(merged_yaml))

## 4 — Train Vehicle Detector (YOLO)


In [ ]:
from ultralytics import YOLO
import torch

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'CUDA: {torch.version.cuda}')

# Load base model
model = YOLO(MODEL_BASE)

# Use actual merged YAML or COCO8 fallback
DATA_YAML = str(merged_yaml_path) if train_imgs else str(fallback_yaml)
print(f'Training data: {DATA_YAML}')

# ── TRAIN ──────────────────────────────────────────────────────────────
results = model.train(
    data     = DATA_YAML,
    **TRAIN_CFG,
)

## 5 — KPI Validation (PRD Section 3.2)


In [ ]:
from ultralytics import YOLO
from pathlib import Path

best_pt = RUNS_DIR / 'jsep' / 'vehicle_detector_v1' / 'weights' / 'best.pt'
print(f'Best weights: {best_pt} — exists: {best_pt.exists()}')

if best_pt.exists():
    val_model = YOLO(str(best_pt))
    metrics = val_model.val(data=DATA_YAML, device=0)

    kpis = {
        'KPI-01 mAP50'      : (metrics.box.map50,  0.90),
        'KPI-04 Precision'  : (metrics.box.mp,     0.85),
        'KPI-05 Recall'     : (metrics.box.mr,     0.78),
    }

    print('\n' + '='*50)
    print('JSEP KPI Validation Results')
    print('='*50)
    all_pass = True
    for name, (val, target) in kpis.items():
        passed = val >= target
        all_pass = all_pass and passed
        mark = '✓ PASS' if passed else '✗ FAIL'
        print(f'  {name}: {val:.4f}  {mark} (target >= {target})')
    print('='*50)
    print(f'Overall: {"ALL KPIs PASSED ✓" if all_pass else "SOME KPIs FAILED ✗"}')

## 6 — Export & Save Artifacts


In [ ]:
import shutil
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/jsep_vehicle_detector_output')
OUTPUT_DIR.mkdir(exist_ok=True)

best_pt = RUNS_DIR / 'jsep' / 'vehicle_detector_v1' / 'weights' / 'best.pt'
last_pt = RUNS_DIR / 'jsep' / 'vehicle_detector_v1' / 'weights' / 'last.pt'

# Copy weights
if best_pt.exists():
    shutil.copy2(best_pt, OUTPUT_DIR / 'vehicle_detector_best.pt')
    print(f'Saved best.pt  → {OUTPUT_DIR / "vehicle_detector_best.pt"}')

if last_pt.exists():
    shutil.copy2(last_pt, OUTPUT_DIR / 'vehicle_detector_last.pt')
    print(f'Saved last.pt  → {OUTPUT_DIR / "vehicle_detector_last.pt"}')

# Copy training results CSV and plots
results_csv = RUNS_DIR / 'jsep' / 'vehicle_detector_v1' / 'results.csv'
if results_csv.exists():
    shutil.copy2(results_csv, OUTPUT_DIR / 'training_results.csv')
    print(f'Saved results.csv')

# Also export as ONNX for inference portability
if best_pt.exists():
    export_model = YOLO(str(best_pt))
    export_model.export(format='onnx', dynamic=True)
    onnx_path = best_pt.parent / 'best.onnx'
    if onnx_path.exists():
        shutil.copy2(onnx_path, OUTPUT_DIR / 'vehicle_detector_best.onnx')
        print(f'Saved best.onnx → {OUTPUT_DIR / "vehicle_detector_best.onnx"}')

print(f'\nAll artifacts saved to: {OUTPUT_DIR}')
print('\nFiles:')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

## 7 — Training Curves Visualization
> Plot mAP50, box loss, cls loss curves from results.csv


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_csv = RUNS_DIR / 'jsep' / 'vehicle_detector_v1' / 'results.csv'
if not results_csv.exists():
    print('No results.csv found. Training may not be complete.')
else:
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]  # clean whitespace

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('JSEP Vehicle Detector — Training Curves', fontsize=14, fontweight='bold')

    # Try to find matching columns robustly
    def plot_col(ax, df, col_patterns, label, color):
        for pat in col_patterns:
            matches = [c for c in df.columns if pat.lower() in c.lower()]
            if matches:
                ax.plot(df['epoch'], df[matches[0]], color=color, label=label)
                ax.set_xlabel('Epoch'); ax.set_ylabel(label)
                ax.set_title(label); ax.legend(); ax.grid(alpha=0.3)
                return
        ax.text(0.5, 0.5, f'{label}\n(not available)', ha='center', va='center', transform=ax.transAxes)

    plot_col(axes[0,0], df, ['train/box_loss', 'box_loss'],  'Train Box Loss',    '#e74c3c')
    plot_col(axes[0,1], df, ['train/cls_loss', 'cls_loss'],  'Train Cls Loss',    '#e67e22')
    plot_col(axes[0,2], df, ['train/dfl_loss', 'dfl_loss'],  'Train DFL Loss',    '#e91e63')
    plot_col(axes[1,0], df, ['metrics/mAP50', 'mAP50'],      'Val mAP@50',        '#27ae60')
    plot_col(axes[1,1], df, ['metrics/precision', 'precision'], 'Val Precision',  '#2980b9')
    plot_col(axes[1,2], df, ['metrics/recall', 'recall'],    'Val Recall',        '#8e44ad')

    # KPI reference lines
    axes[1,0].axhline(0.90, ls='--', color='black', alpha=0.5, label='KPI-01 target')
    axes[1,1].axhline(0.85, ls='--', color='black', alpha=0.5, label='KPI-04 target')
    axes[1,2].axhline(0.78, ls='--', color='black', alpha=0.5, label='KPI-05 target')
    for ax in axes[1]:
        ax.legend(fontsize=8)

    plt.tight_layout()
    fig.savefig('/kaggle/working/jsep_vehicle_detector_output/training_curves.png', dpi=150)
    plt.show()
    print('Training curves saved.')

---
## How to deploy the trained weights to the JSEP server
```bash
# 1. Download vehicle_detector_best.pt from Kaggle Output
# 2. SCP / copy to the server:
scp vehicle_detector_best.pt user@jsep-server:/home/mghiffaa/Jsep/models/
# 3. Update .env:
DETECTION_MODEL=models/vehicle_detector_best.pt
# 4. Restart the pipeline worker
```
